# Gen3 Embeddings Demo

> This will demonstrate how to create and retrieve embeddings in bulk from Gen3

First, let's install the Gen3 Python Software Development Kit (SDK), which includes a command line interface (CLI).

In [ ]:
%pip install --upgrade pip
# %pip install gen3 --upgrade

In [ ]:
# we also need pandas for some nice visualizations of output files
import pandas as pd

If you are running Gen3 locally, you can set some variables to point to the right credentials file.

If you are trying to interact with a production instance, just leave the default `ai`. e.g. don't uncomment - the default credentials setup should point you to the right place if you have your API key in the default location `~/.gen3/credentials.json`.

In [ ]:
url_prefix = "https://foobar.dev.planx-pla.net/ai"
auth = "--auth ~/.gen3/local_helm_test_user.json"

# url_prefix = "https://foobar-local.dev.planx-pla.net/ai"
# auth = ""

## Gen3 AI Embeddings CLI

In [ ]:
!gen3 ai embeddings --help

In [ ]:
!gen3 ai embeddings collections --help

Get a response from the service by reading collections. 

> IMPORTANT: You need appropriate permissions to read (and for future sections: write).
> So these commands may be empty or fail unless you have those authorizations in the environment your credentials are for.

In [ ]:
!gen3 $auth ai embeddings collections read

## Create Embeddings Collections

This will show creation of collections and embeddings. So you need write permission or this will fail. You can run the Gen3 Embeddings API locally to test this out. See the [Gen3 AI repo README](https://github.com/uc-cdis/gen3-ai) for more information.

In [ ]:
!gen3 $auth ai embeddings collections delete "ctds-github-md"
!gen3 $auth ai embeddings collections create "ctds-github-md" --dimensions 384 --description "All markdown from CTDS Github"

In [ ]:
!gen3 $auth ai embeddings collections read "ctds-github-md" 

In [ ]:
# try to delete collections that might already exist
!gen3 $auth ai embeddings collections delete "test_expr"
!gen3 $auth ai embeddings collections delete "test_hist"
!gen3 $auth ai embeddings collections delete "test_summ"

Let's create some more example collections.

> IMPORTANT: You need permission to create and manage these collections *before* running the commands. So ensure the Gen3 operator adds these resources to the `user.yaml` and provides your user permission to them. If you are running Gen3 yourself, you can see the [Gen3 AI repo README](https://github.com/uc-cdis/gen3-ai) for more information on how to set up appropriate auth.

In [ ]:
!gen3 $auth ai embeddings collections create "test_expr" --dimensions 256 --description "test expr data"
!gen3 $auth ai embeddings collections create "test_hist" --dimensions 1536 --description "test hist data"

You can also create collections of larger dimensional size. This will use a `vector_type` of `halfvec` to fit it into the underlying database.

In [ ]:
!gen3 $auth ai embeddings collections create "test_summ" --dimensions 4096 --description "test summ data"

## Publish Data into Embeddings Collections

Now that we have created collections for embeddings, we can publish the actual embeddings into those collections.

To do this, you need a Gen3 Embeddings Manifests. Conveniently, there are examples in the Gen3 Python SDK/CLI repo in the tests folder you can use.

In [ ]:
!gen3 ai embeddings publish --help

See the above help message to understand what we need to publish embeddings. 

**tl;dr** we need a manifest file with a row per embedding. That row needs to contain the vector (embedding), along with other metadata.

In [ ]:
# here's a quick visualization of the columns/data of this input manifest
df = pd.read_csv('../../tests/embeddings_tests/test_expr.tsv', sep='\t', nrows=10)
df

In [ ]:
!gen3 $auth ai embeddings publish ../../tests/embeddings_tests/test_expr.tsv --default-collection test_expr --batch-size 50

In [ ]:
# here's a quick visualization of the columns/data of this input manifest
df = pd.read_csv('../../tests/embeddings_tests/test_hist.tsv', sep='\t', nrows=10)
df

In [ ]:
!gen3 $auth ai embeddings publish ../../tests/embeddings_tests/test_hist.tsv --default-collection test_hist --batch-size 20

In [ ]:
# here's a quick visualization of the columns/data of this input manifest
df = pd.read_csv('../../tests/embeddings_tests/test_summ.tsv', sep='\t', nrows=10)
df

In [ ]:
!gen3 $auth ai embeddings publish ../../tests/embeddings_tests/test_summ.tsv --default-collection test_summ --batch-size 10 --overwrite

## Convert Published Embeddings Manifests into Indexing Manifests

Each of the `publish` commands above generated an output file `{input_filename}_output.tsv` (unless you overrode the output filename). 

Those outputs are manifests that now contain the final `embedding_id` and other Gen3 Embedding information that came back from creating embeddings through the Gen3 Embeddings API.

We can **convert** those _outputted_ Published Gen3 Embeddings Manifests into Gen3 Indexing Manifests to _input_ into the Gen3 Indexing process. This will allow us to create persistent, indexed records with globally unique identifiers (GUIDs) in Gen3 through the Gen3 Indexing API.

So first, let's convert to the expected format for indexing.

In [ ]:
# here's a quick visualization of the columns/data of the output manifest from previous `publish` commands
# this is what we'll convert
df = pd.read_csv('../../tests/embeddings_tests/test_expr_output.tsv', sep='\t', nrows=3)
df

In [ ]:
!gen3 ai embeddings convert --help

In [ ]:
!gen3 $auth ai embeddings convert ../../tests/embeddings_tests/test_expr_output.tsv --url-prefix $url_prefix
!gen3 $auth ai embeddings convert ../../tests/embeddings_tests/test_hist_output.tsv --url-prefix $url_prefix
!gen3 $auth ai embeddings convert ../../tests/embeddings_tests/test_summ_output.tsv --url-prefix $url_prefix

That `convert` command created new `{original_filename}_converted.tsv` files. Let's take a look at those:

In [ ]:
df = pd.read_csv('../../tests/embeddings_tests/test_expr_output_converted.tsv', sep='\t', nrows=3)
df

## Create Gen3 Indexed Records with the Indexing Manifest

The manifest we converted to above is a Gen3 Indexing Manifest with no GUIDs - we'll let the Gen3 Indexing command and backend generate those for us.

The `md5` checksum and `size` in bytes columns above are of the JSON-stringified version of the vector (in other words, the "data" is the vector itself).

`url` is a direct link to the Gen3 Embeddings API for that particular embedding.

Now, before we index, let's validate our Gen3 Indexing manifest is correctly formatted:

In [ ]:
!gen3 objects manifest validate-manifest-format --help

In [ ]:
!gen3 objects manifest validate-manifest-format ../../tests/embeddings_tests/test_expr_output_converted.tsv --allowed-protocols "https http"
!gen3 objects manifest validate-manifest-format ../../tests/embeddings_tests/test_hist_output_converted.tsv --allowed-protocols "https http"
!gen3 objects manifest validate-manifest-format ../../tests/embeddings_tests/test_summ_output_converted.tsv --allowed-protocols "https http"

In [ ]:
!gen3 objects manifest publish --help

> IMPORTANT NOTE: Publishing the manifest through the Gen3 Indexing API requires permissions to write to that API. 

**For Gen3 Operators**: typically there is an `indexd_admin` policy. One option to provide the necessary permissions is to add the `/vectorstore/collection/` resource path to that. This will allow full administrative management of indexing records with `authz` beginning with that path.

Example `user.yaml` snippet:

```yaml
...
      - id: indexd_admin
        description: full access to indexd API
        role_ids:
          - indexd_admin
        resource_paths:
          - /programs
          - /vectorstore/collections       <----- THIS IS NEW
...
```

In [ ]:
!gen3 $auth -v objects manifest publish ../../tests/embeddings_tests/test_expr_output_converted.tsv --out-manifest-file ../../tests/embeddings_tests/test_expr_output_converted_indexed.tsv --thread-num 1

In [ ]:
!gen3 $auth -v objects manifest verify ../../tests/embeddings_tests/test_expr_output_converted_indexed.tsv --max-concurrent-requests 1

> Note: if the above command fails, you can try to move on anyway. It is just to double-check that the manifest entries got created - which may be true even if the verify command fails. We plan on addressing this flakiness in a future update.

In [ ]:
!gen3 $auth -v objects manifest publish ../../tests/embeddings_tests/test_hist_output_converted.tsv --out-manifest-file ../../tests/embeddings_tests/test_hist_output_converted_indexed.tsv --thread-num 1

In [ ]:
!gen3 $auth -v objects manifest verify ../../tests/embeddings_tests/test_hist_output_converted_indexed.tsv --max-concurrent-requests 1

In [ ]:
!gen3 $auth -v objects manifest publish ../../tests/embeddings_tests/test_summ_output_converted.tsv --out-manifest-file ../../tests/embeddings_tests/test_summ_output_converted_indexed.tsv --thread-num 1

In [ ]:
!gen3 $auth -v objects manifest verify ../../tests/embeddings_tests/test_summ_output_converted_indexed.tsv --max-concurrent-requests 1

Now there are indexed records with GUIDs. The above tool output an `*_indexed.tsv` manifest. We can take a look and see that now the `guid` column is filled out! 

In [ ]:
df = pd.read_csv('../../tests/embeddings_tests/test_expr_output_converted_indexed.tsv', sep='\t', nrows=3)
df

## Everything is in Gen3 now!

Let's recap. We have:

- Created Gen3 Embeddings Collections to store embeddings of different dimensionality
- Ingested Embeddings Manifests of embeddings through the Gen3 Embeddings API into the collections we created
- Converted output manifest from embedding creation into indexing manifests
- Ingested Indexing Manifests to create Gen3 Indexed Records with assigned GUIDs

At this point, we have the embeddings themselves stored and we've created persistent identifiers (GUIDs). 

Now you can use those GUIDs the same way you use other Gen3 GUIDs for files. So if you have a Gen3 Data Model with nodes that point to sample files identified by GUIDs, you can now have a new "embedding" node with a GUID pointing to the embedding!

This will allow traditional Gen3 search for data with embeddings.

If you want to search the embeddings collections themselves, the Gen3 Embedding API exposes search functionality as well. This supports similarity search.

## Bulk Retrieval of Gen3 Embeddings via their Indexed GUIDs

Let's assume we've found GUIDs of interest via a search in Gen3 (and for simplicity, let's assume those GUIDs are ALL the GUIDs we have indexed previously in this notebook). Now, given **only** those GUIDs, we want to bulk retrieve the actual embeddings from Gen3 to do some AI/ML analysis.

In [ ]:
"""
This just aggregates all the GUIDs from all our indexed files into a single file.

This output file simulates (or rather, skips) the process of finding data of interest and getting the GUIDs.
"""
import glob
import pandas as pd

file_pattern = "../../tests/embeddings_tests/test_*_output_converted_indexed.tsv"
output_file = "../../tests/embeddings_tests/test_aggregated_guids.tsv"

all_files = glob.glob(file_pattern)

df_list = []
for file in all_files:
    # usecols ensures we don't waste memory. only load guids
    df = pd.read_csv(file, sep="\t", usecols=["guid"])
    df_list.append(df)

combined_df = pd.concat(df_list, ignore_index=True)
combined_df.to_csv(output_file, sep="\t", index=False)

print(f"Done! Aggregated {len(all_files)} files into: {output_file}")

In [ ]:
df = pd.read_csv('../../tests/embeddings_tests/test_aggregated_guids.tsv', sep='\t')
df

## Bulk retrieve embeddings from Fence

Now we can pretend we found a bunch of GUIDs of interest in Gen3.

For this example, we'll use all the GUIDs we previously created.

By providing a manifest of only GUIDs (or passing on the CLI), we can get the _content_ of the referenced data from the indexed record, e.g. the embedding itself.


In [ ]:
from gen3.file import Gen3File
from gen3.auth import Gen3Auth

import time

start = time.perf_counter()

#auth = Gen3Auth(refresh_file="FULL_PATH_TO/creds.json")
auth = Gen3Auth(refresh_file="FULL_PATH_TO/local_helm_test_user.json")
gen3_file = Gen3File(auth.endpoint, auth_provider=auth)

embeddings_contents = gen3_file.get_bulk_content(input_file="../../tests/embeddings_tests/test_aggregated_guids.tsv")
 
end = time.perf_counter()
print(f"Total runtime: {end - start:.3f} seconds")

print(f"Got all {len(embeddings_contents)} GUIDs")

import pandas as pd

rows = []
for i, (guid, emb) in enumerate(embeddings_contents.items()):
    if i >= 100:  # sample first 100 to keep it light
        break
    rows.append({
        "guid": emb.guid,
        "embedding_id": emb.embedding_id,
        "embedding_len": emb.embedding.shape[0],
        "dtype": str(emb.embedding.dtype),
        "authz": emb.authz,
        "collection_id": emb.collection_id,
        **emb.metadata
    })

df_sample = pd.DataFrame(rows)
df_sample.head()

# for guid, embedding_content in embeddings_contents.items():
#     # the GUIDs are already an efficient numpy array!
#     print(type(embedding_content.embedding))
#     print(embedding_content)
#     break


# do other AI/ML work with the embeddings!

We have shown the full flow from original vectors -> storage in Gen3 -> retrieval from Gen3 in an efficient bulk pipeline. 

We've also shown that Gen3 Embeddings can be indexed like files to provide GUIDs which can be referenced and searched the same way file-based GUIDs are today.